In [ ]:
import os
import csv
import clip
import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "ViT-B/32"

FEATURE_FILE = "/content/drive/MyDrive/clip_features_products/clip_features.npz"
CSV_FILE = "/content/products_with_subcategory.csv"
QUERY_IMAGE = "/content/elegantiski-juodi-kojines-tipo-ilgaauliai-batai-des910s-black.jpg"
TOP_N = 13

# Leidžiame rinktis ar norima filtruoti pagal lytį
CATEGORY_FILTER = input("Įveskite 'Moterims', 'Vyrams' arba palikite tuščią: ").strip() or None

# Modelio paleidimas
model, preprocess = clip.load(MODEL_NAME, device=DEVICE)
model.eval()

# Turimų požymių paleidimas
data = np.load(FEATURE_FILE, allow_pickle=True)
image_paths = data["paths"]
features = data["features"]

# Susidaromas žodynas
products_dict = {}
with open(CSV_FILE, encoding="utf-8") as f:
    reader = csv.DictReader(f, delimiter=";")
    for row in reader:
        prod_id = str(row["ID"])
        products_dict[prod_id] = row

# Atfiltruojama pagal kategoriją
filtered_features = []
filtered_image_paths = []

for idx, path in enumerate(image_paths):
    prod_id = os.path.splitext(os.path.basename(path))[0]
    row = products_dict.get(prod_id, {})
    if CATEGORY_FILTER is None or row.get("Category") == CATEGORY_FILTER:
        filtered_features.append(features[idx])
        filtered_image_paths.append(path)

if not filtered_features:
    raise ValueError("Nėra produktų šiai kategorijai")

filtered_features = np.vstack(filtered_features)

# Apdorojamas vaizdas
image = Image.open(QUERY_IMAGE).convert("RGB")
image_input = preprocess(image).unsqueeze(0).to(DEVICE)

with torch.no_grad():
    query_feature = model.encode_image(image_input)
    query_feature = query_feature / query_feature.norm(dim=-1, keepdim=True) # normavimas

query_feature = query_feature.cpu().numpy()

# Atliekama panašaus vaizdo paieška pagal vaizdą
nn = NearestNeighbors(n_neighbors=TOP_N, metric="cosine")
nn.fit(filtered_features)

distances, top_idx = nn.kneighbors(query_feature)

# Iš atstumų paverčiame į panašumus
similarities = 1 - distances[0]

# Šiuo atveju išspausdinami rezultatai teksto pavidalu
print(f"\nTop {TOP_N} similar products to {QUERY_IMAGE}:\n")
for rank, (idx, sim) in enumerate(zip(top_idx[0], similarities), start=1):
    filename = os.path.basename(filtered_image_paths[idx])
    prod_id = os.path.splitext(filename)[0]
    row = products_dict.get(prod_id, {})
    subcat = row.get("Subcategory", "Unknown")
    price = row.get("Price", "Unknown")
    category = row.get("Category", "Unknown")

    print(f"{rank}. {filename} | Category: {category} | Subcategory: {subcat} | Price: {price} | Similarity: {sim:.4f}")

# Vaizdų rezultatai
plt.figure(figsize=(4 * TOP_N, 5))
for i, idx in enumerate(top_idx[0]):
    img = Image.open(filtered_image_paths[idx]).convert("RGB")
    filename = os.path.basename(filtered_image_paths[idx])
    prod_id = os.path.splitext(filename)[0]
    row = products_dict.get(prod_id, {})
    subcat = row.get("Subcategory", "Unknown")
    price = row.get("Price", "Unknown")

    plt.subplot(1, TOP_N, i + 1)
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"{subcat}\n{price}", fontsize=9)

plt.show()
